# MangoScan Model 0 — Mango/Stem Detector

Trains a YOLO11n detector that finds mangoes (and their stems) in a raw
camera frame, so Models 1–4 receive a cropped, centred fruit instead of a
whole conveyor.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Upload the *prepared* dataset to Drive at
`MyDrive/MangoScan/data/detector_dataset/` — the tree written by
`tools/prepare_detector_dataset.py`, **not** the 17 GB original.

In [ ]:
!nvidia-smi

## 0. Mount Drive and sanity-check the dataset

If image and label counts disagree for any split, stop — Ultralytics will
train happily on a mismatched pair and the metrics will be quietly wrong.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DATA = pathlib.Path('/content/drive/MyDrive/MangoScan/data/detector_dataset')
assert DATA.is_dir(), f'Not found: {DATA}. Upload the prepared dataset there first.'
assert (DATA / 'data.yaml').is_file(), 'data.yaml missing - re-run prepare_detector_dataset.py'

for split in ('train', 'val', 'test'):
    n_img = len(list((DATA / 'images' / split).glob('*.jpg')))
    n_lbl = len(list((DATA / 'labels' / split).glob('*.txt')))
    flag = '' if n_img == n_lbl else '   <-- MISMATCH, stop here'
    print(f'{split:<6} {n_img:>5} images  {n_lbl:>5} labels{flag}')

print()
print((DATA / 'data.yaml').read_text())

## 1. Install, and get the code from Drive

**Not** `git clone`. `MyTasks/` is in `.gitignore` and has never been
committed, so the GitHub repo does not contain this notebook's own scripts.
(The Model 1 notebook still clones, and is broken for exactly this reason.)

Upload the `model0_detector/` folder to
`MyDrive/MangoScan/code/model0_detector/`, alongside the dataset.

In [ ]:
!pip install -q ultralytics

import shutil

CODE = pathlib.Path('/content/drive/MyDrive/MangoScan/code/model0_detector')
assert CODE.is_dir(), f'Not found: {CODE}. Upload model0_detector/ there.'

# Copy out of Drive rather than working in it: training writes to runs/ on
# every epoch, and Drive's FUSE mount makes that slow enough to matter over
# 150 epochs. Cell 7 copies the weights back.
local = pathlib.Path('/content/model0_detector')
if local.exists():
    shutil.rmtree(local)
shutil.copytree(CODE, local)
%cd /content/model0_detector
!ls -R | head -20

## 2. Point `data.yaml` at Drive

`data.yaml` stores an absolute `path`, written on the machine that prepared
the dataset. Colab sees a different filesystem, so rewrite it.

In [ ]:
import re

yaml_path = DATA / 'data.yaml'
text = re.sub(r'^path: .*$', f'path: {DATA.as_posix()}', yaml_path.read_text(), flags=re.M)
yaml_path.write_text(text)
print(text)

## 3. Train

150 epochs on a T4 over ~1,700 images takes roughly 1.5–3 hours. Early
stopping fires after 30 epochs without improvement, so it usually finishes
sooner.

Watch `metrics/recall(B)` more than the loss curves — see the note at the
top of `scripts/03_evaluate.py` for why recall is the number that matters.

In [ ]:
!python scripts/02_train.py --config configs/detector.toml \
    --data /content/drive/MyDrive/MangoScan/data/detector_dataset/data.yaml

## 4. Evaluate on the held-out test split

**mango recall** is the number to defend. A missed mango is a fruit that
never reaches Models 1–4 and never gets routed at all — strictly worse than
one routed to the wrong bin.

Stem numbers will be lower, and that is expected: the stem is small, often
occluded, and supplies orientation only. It is not on the routing path, so
do not tune against it.

In [ ]:
!python scripts/03_evaluate.py --config configs/detector.toml \
    --weights runs/model0_detector/weights/best.pt \
    --data /content/drive/MyDrive/MangoScan/data/detector_dataset/data.yaml

## 5. Look at actual predictions

Metrics hide failure modes. Look at the boxes. You are checking for two
things: mangoes with no box at all, and boxes loose enough to include
conveyor background — Model 4 reads colour off those pixels.

In [ ]:
import random

import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO('runs/model0_detector/weights/best.pt')
test_images = sorted((DATA / 'images' / 'test').glob('*.jpg'))
sample = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, img in zip(axes.ravel(), sample):
    result = model.predict(str(img), imgsz=640, conf=0.25, verbose=False)[0]
    ax.imshow(result.plot()[:, :, ::-1])
    ax.set_title(f'{img.name}  ({len(result.boxes)} boxes)', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Export for deployment

In [ ]:
from ultralytics import YOLO

model = YOLO('runs/model0_detector/weights/best.pt')
model.export(format='onnx', opset=12, simplify=True, imgsz=640)

## 7. Save the weights back to Drive

Colab discards `/content` when the runtime recycles. Do not skip this cell.

In [ ]:
import pathlib
import shutil

dest = pathlib.Path('/content/drive/MyDrive/MangoScan/models/model0_detector')
dest.mkdir(parents=True, exist_ok=True)
src = pathlib.Path('runs/model0_detector/weights')
for f in sorted(src.glob('*.pt')) + sorted(src.glob('*.onnx')) + sorted(src.glob('*.json')):
    shutil.copy2(f, dest / f.name)
    print('saved', dest / f.name)